In [1]:
import pandas as pd
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# We will download a clean version of the dataset from a raw GitHub link
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Let's look at the first 5 rows
print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


Data Pre-processing

In [3]:
# Convert labels to numbers: 'ham' = 0, 'spam' = 1
df['label_num'] = df.label.map({'ham':0, 'spam':1})

# Define our features (X) and our target (y)
X = df['message']
y = df['label_num']

# Split the data into Training Data (80%) and Testing Data (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training messages: {len(X_train)}")
print(f"Testing messages: {len(X_test)}")

Training messages: 4457
Testing messages: 1115


Convert Text to Numbers (Vectorization)

In [4]:
# Initialize the Vectorizer
vectorizer = TfidfVectorizer(stop_words='english')

# "Fit" it on the training data (learn the vocabulary) and transform it into numbers
X_train_transformed = vectorizer.fit_transform(X_train)

# ONLY transform the testing data (do NOT fit it, we only use the vocabulary learned from training)
X_test_transformed = vectorizer.transform(X_test)

Train the Model

In [5]:
# Initialize the Naive Bayes model
model = MultinomialNB()

# Train the model
model.fit(X_train_transformed, y_train)

print("Model training complete!")

Model training complete!


Test and Evaluate the Model

In [6]:
# Make predictions on the test data
predictions = model.predict(X_test_transformed)

# Calculate accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

# Detailed performance report
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=['Ham', 'Spam']))

Model Accuracy: 97.85%

Classification Report:
              precision    recall  f1-score   support

         Ham       0.98      1.00      0.99       966
        Spam       1.00      0.84      0.91       149

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



Test it with your own messages!

In [7]:
def check_message(message_text):
    # 1. Transform the text into numbers using the same vectorizer
    transformed_text = vectorizer.transform([message_text])

    # 2. Make a prediction
    prediction = model.predict(transformed_text)[0]

    # 3. Output the result
    if prediction == 1:
        print(f"🚨 SPAM DETECTED: '{message_text}'")
    else:
        print(f"✅ SAFE (HAM): '{message_text}'")

# Try it out!
check_message("Hey, are we still on for lunch tomorrow?")
check_message("URGENT! You have won a 1 week FREE membership in our $100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010")

✅ SAFE (HAM): 'Hey, are we still on for lunch tomorrow?'
🚨 SPAM DETECTED: 'URGENT! You have won a 1 week FREE membership in our $100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010'
